# Hotel / Airbnb Booking Pricing & Occupancy Analytics

## Step 1: Import Libraries

In [ ]:
import os

# Matplotlib needs a writable folder for its settings and font cache.
# This keeps the project friendly on computers where the home folder is restricted.
if "MPLCONFIGDIR" not in os.environ:
    os.environ["MPLCONFIGDIR"] = os.path.join(os.getcwd(), ".matplotlib_cache")

if "XDG_CACHE_HOME" not in os.environ:
    os.environ["XDG_CACHE_HOME"] = os.path.join(os.getcwd(), ".cache")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 2: Set Folder Paths

In [ ]:
raw_folder_path = "data/raw"
processed_folder_path = "data/processed"
images_folder_path = "images"

if os.path.exists(raw_folder_path) == False:
    raw_folder_path = "../data/raw"
    processed_folder_path = "../data/processed"
    images_folder_path = "../images"

print("Raw data folder:", raw_folder_path)
print("Processed data folder:", processed_folder_path)
print("Images folder:", images_folder_path)

## Step 3: Find and Load the Dataset

In [ ]:
data_file_path = None

if os.path.exists(raw_folder_path):
    file_names = os.listdir(raw_folder_path)

    for file_name in file_names:
        lower_file_name = file_name.lower()

        if lower_file_name.endswith(".csv"):
            data_file_path = os.path.join(raw_folder_path, file_name)
            break

        if lower_file_name.endswith(".xlsx") or lower_file_name.endswith(".xls"):
            data_file_path = os.path.join(raw_folder_path, file_name)
            break

if data_file_path is None:
    print("No CSV or Excel file was found in data/raw/.")
else:
    print("Dataset found:", data_file_path)

In [ ]:
booking_data = None

if data_file_path is not None:
    lower_file_path = data_file_path.lower()

    if lower_file_path.endswith(".csv"):
        booking_data = pd.read_csv(data_file_path)
        print("The dataset was loaded as a CSV file.")

    if lower_file_path.endswith(".xlsx") or lower_file_path.endswith(".xls"):
        booking_data = pd.read_excel(data_file_path)
        print("The dataset was loaded as an Excel file.")

## Step 4: Understand the Data

In [ ]:
if booking_data is not None:
    print("FIRST 5 ROWS")
    display(booking_data.head())
    print("These rows show what the dataset looks like at the beginning.")

    print("LAST 5 ROWS")
    display(booking_data.tail())
    print("These rows show what the dataset looks like at the end.")

    print("DATA SHAPE")
    print(booking_data.shape)
    print("The first number is rows. The second number is columns.")

    print("COLUMN NAMES")
    print(booking_data.columns)
    print("These are the available fields in the dataset.")

In [ ]:
if booking_data is not None:
    print("DATA TYPES")
    print(booking_data.dtypes)
    print("Data types help us know which columns are text, numbers, or dates.")

    print("MISSING VALUES")
    print(booking_data.isnull().sum())
    print("Missing values show where information is not available.")

    print("DUPLICATE ROWS")
    print(booking_data.duplicated().sum())
    print("Duplicate rows may repeat the same booking more than once.")

    print("BASIC STATISTICS")
    display(booking_data.describe(include="all"))
    print("Basic statistics summarize numeric and text columns.")

## Step 5: Data Cleaning

In [ ]:
if booking_data is not None:
    cleaned_data = booking_data.copy()

    print("Cleaning step: Remove duplicate rows")
    print("Why: Duplicate rows can count the same booking more than once.")
    rows_before = cleaned_data.shape[0]
    cleaned_data = cleaned_data.drop_duplicates()
    rows_after = cleaned_data.shape[0]
    print("Rows before:", rows_before)
    print("Rows after:", rows_after)
    print("Duplicate rows removed:", rows_before - rows_after)

In [ ]:
if booking_data is not None:
    print("Cleaning step: Clean text columns")
    print("Why: Extra spaces and different letter cases can create duplicate categories.")

    for column_name in cleaned_data.columns:
        column_type = str(cleaned_data[column_name].dtype)

        if column_type == "object" or column_type == "str" or "string" in column_type:
            cleaned_data[column_name] = cleaned_data[column_name].astype(str)
            cleaned_data[column_name] = cleaned_data[column_name].str.strip()
            cleaned_data[column_name] = cleaned_data[column_name].str.title()

            missing_text_mask = cleaned_data[column_name] == "Nan"
            cleaned_data.loc[missing_text_mask, column_name] = pd.NA

    print("Text columns were cleaned.")

In [ ]:
if booking_data is not None:
    print("Cleaning step: Convert date columns")
    print("Why: Date columns should be in date format for monthly and seasonal analysis.")

    for column_name in cleaned_data.columns:
        lower_column_name = column_name.lower()
        is_full_date_column = False

        if lower_column_name == "reservation_status_date":
            is_full_date_column = True

        if lower_column_name == "booking_date":
            is_full_date_column = True

        if lower_column_name == "arrival_date":
            is_full_date_column = True

        if is_full_date_column:
            cleaned_data[column_name] = pd.to_datetime(cleaned_data[column_name], errors="coerce")
            print("Converted to date:", column_name)

In [ ]:
if booking_data is not None:
    print("Cleaning step: Convert numeric columns")
    print("Why: Numeric columns should be numbers so Python can calculate totals and averages.")

    possible_numeric_columns = [
        "adr",
        "price",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "adults",
        "children",
        "babies",
        "lead_time",
        "days_in_waiting_list",
        "is_canceled"
    ]

    for column_name in possible_numeric_columns:
        if column_name in cleaned_data.columns:
            cleaned_data[column_name] = pd.to_numeric(cleaned_data[column_name], errors="coerce")
            print("Converted to numeric:", column_name)

In [ ]:
if booking_data is not None:
    print("Cleaning step: Handle missing values")
    print("Why: Missing values can break calculations and charts.")

    for column_name in cleaned_data.columns:
        missing_count = cleaned_data[column_name].isnull().sum()

        if missing_count > 0:
            column_type = str(cleaned_data[column_name].dtype)

            if column_type == "object" or column_type == "str" or "string" in column_type:
                cleaned_data[column_name] = cleaned_data[column_name].fillna("Unknown")
                print(column_name, "- filled missing text values with Unknown.")
            elif "int" in column_type or "float" in column_type:
                median_value = cleaned_data[column_name].median()
                cleaned_data[column_name] = cleaned_data[column_name].fillna(median_value)
                print(column_name, "- filled missing numeric values with median.")
            elif "datetime" in column_type:
                most_common_value = cleaned_data[column_name].mode()

                if len(most_common_value) > 0:
                    cleaned_data[column_name] = cleaned_data[column_name].fillna(most_common_value[0])
                    print(column_name, "- filled missing date values with most common date.")
            else:
                cleaned_data[column_name] = cleaned_data[column_name].fillna("Unknown")
                print(column_name, "- filled missing values with Unknown.")

In [ ]:
if booking_data is not None:
    print("Cleaning step: Remove impossible values")
    print("Why: Negative prices, negative nights, and negative guest counts do not make business sense.")

    rows_before = cleaned_data.shape[0]

    positive_columns = [
        "adr",
        "price",
        "average_daily_rate",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "adults",
        "children",
        "babies",
        "lead_time"
    ]

    for column_name in positive_columns:
        if column_name in cleaned_data.columns:
            cleaned_data = cleaned_data[cleaned_data[column_name] >= 0]
            print("Checked non-negative values for:", column_name)

    rows_after = cleaned_data.shape[0]
    print("Rows removed:", rows_before - rows_after)
    print("Final cleaned data shape:", cleaned_data.shape)

## Step 6: Feature Engineering

We create features only when the required columns exist. If columns are missing, we skip that feature and print a clear message.

In [ ]:
if booking_data is not None:
    if "stays_in_weekend_nights" in cleaned_data.columns and "stays_in_week_nights" in cleaned_data.columns:
        cleaned_data["stay_length"] = cleaned_data["stays_in_weekend_nights"] + cleaned_data["stays_in_week_nights"]
        print("Created feature: stay_length")
    else:
        print("Skipped stay_length because required night columns are missing.")

    date_column = None

    if "reservation_status_date" in cleaned_data.columns:
        date_column = "reservation_status_date"
    elif "booking_date" in cleaned_data.columns:
        date_column = "booking_date"
    elif "arrival_date" in cleaned_data.columns:
        date_column = "arrival_date"

    if date_column is not None:
        cleaned_data[date_column] = pd.to_datetime(cleaned_data[date_column], errors="coerce")
        cleaned_data["booking_month"] = cleaned_data[date_column].dt.month
        cleaned_data["booking_year"] = cleaned_data[date_column].dt.year
        cleaned_data["booking_weekday"] = cleaned_data[date_column].dt.day_name()
        print("Created booking month, year, and weekday from:", date_column)
    elif "arrival_date_month" in cleaned_data.columns and "arrival_date_year" in cleaned_data.columns:
        cleaned_data["booking_month"] = cleaned_data["arrival_date_month"]
        cleaned_data["booking_year"] = cleaned_data["arrival_date_year"]
        print("Created booking month and year from arrival columns.")
    else:
        print("Skipped booking date features because no usable date columns were found.")

In [ ]:
if booking_data is not None:
    if "booking_weekday" in cleaned_data.columns:
        cleaned_data["weekend_booking"] = "No"

        saturday_rows = cleaned_data["booking_weekday"] == "Saturday"
        sunday_rows = cleaned_data["booking_weekday"] == "Sunday"

        cleaned_data.loc[saturday_rows, "weekend_booking"] = "Yes"
        cleaned_data.loc[sunday_rows, "weekend_booking"] = "Yes"
        print("Created feature: weekend_booking")
    else:
        print("Skipped weekend_booking because booking_weekday is missing.")

    if "adr" in cleaned_data.columns and "stay_length" in cleaned_data.columns:
        cleaned_data["revenue"] = cleaned_data["adr"] * cleaned_data["stay_length"]
        print("Created feature: revenue")
    elif "price" in cleaned_data.columns and "stay_length" in cleaned_data.columns:
        cleaned_data["revenue"] = cleaned_data["price"] * cleaned_data["stay_length"]
        print("Created feature: revenue")
    else:
        print("Skipped revenue because price/ADR and stay length columns are missing.")

    if "revenue" in cleaned_data.columns and "is_canceled" in cleaned_data.columns:
        cleaned_data["revenue_lost"] = 0.0
        canceled_rows = cleaned_data["is_canceled"] == 1
        cleaned_data.loc[canceled_rows, "revenue_lost"] = cleaned_data.loc[canceled_rows, "revenue"]
        print("Created feature: revenue_lost")
    else:
        print("Skipped revenue_lost because revenue or is_canceled is missing.")

In [ ]:
if booking_data is not None:
    if "lead_time" in cleaned_data.columns:
        cleaned_data["advance_booking_days"] = cleaned_data["lead_time"]
        print("Created feature: advance_booking_days")
    else:
        print("Skipped advance_booking_days because lead_time is missing.")

    if "is_canceled" in cleaned_data.columns:
        cleaned_data["occupancy_indicator"] = "Occupied"
        canceled_rows = cleaned_data["is_canceled"] == 1
        cleaned_data.loc[canceled_rows, "occupancy_indicator"] = "Not Occupied"
        print("Created feature: occupancy_indicator")
    else:
        print("Skipped occupancy_indicator because is_canceled is missing.")

    if "adr" in cleaned_data.columns:
        cleaned_data["average_daily_rate"] = cleaned_data["adr"]
        print("Created feature: average_daily_rate")
    elif "price" in cleaned_data.columns:
        cleaned_data["average_daily_rate"] = cleaned_data["price"]
        print("Created feature: average_daily_rate")
    else:
        print("Skipped average_daily_rate because adr or price is missing.")

## Step 7: Exploratory Data Analysis

Now we answer simple business questions using the columns that are available.

In [ ]:
if booking_data is not None:
    total_bookings = cleaned_data.shape[0]
    print("Total bookings:", total_bookings)
    print("This shows the total number of booking records in the dataset.")

    if "revenue" in cleaned_data.columns:
        total_revenue = cleaned_data["revenue"].sum()
        print("Total revenue:", total_revenue)
        print("This shows the total estimated revenue from bookings.")

    if "average_daily_rate" in cleaned_data.columns:
        average_price = cleaned_data["average_daily_rate"].mean()
        print("Average daily rate:", average_price)
        print("This shows the average price per night.")

In [ ]:
if booking_data is not None:
    if "booking_month" in cleaned_data.columns:
        monthly_bookings = cleaned_data["booking_month"].value_counts().sort_index()
        print("Monthly bookings")
        print(monthly_bookings)
        print("This shows which months have more bookings.")
    else:
        print("Skipped monthly bookings because booking_month is missing.")

    if "is_canceled" in cleaned_data.columns:
        cancellation_rate = cleaned_data["is_canceled"].mean() * 100
        print("Cancellation rate:", cancellation_rate)
        print("This shows what percentage of bookings were cancelled.")

## Step 8: Visualizations

In [ ]:
if booking_data is not None:
    if os.path.exists(images_folder_path) == False:
        os.makedirs(images_folder_path)

    if "booking_month" in cleaned_data.columns:
        monthly_bookings = cleaned_data["booking_month"].value_counts().sort_index()

        plt.figure(figsize=(10, 5))
        plt.plot(monthly_bookings.index, monthly_bookings.values, marker="o")
        plt.title("Monthly Bookings")
        plt.xlabel("Month")
        plt.ylabel("Number of Bookings")
        plt.tight_layout()
        plt.savefig(os.path.join(images_folder_path, "monthly_bookings.png"))
        plt.show()

        highest_month = monthly_bookings.idxmax()
        print("Observation: Month", highest_month, "has the highest number of bookings.")

In [ ]:
if booking_data is not None:
    if "hotel" in cleaned_data.columns and "revenue" in cleaned_data.columns:
        revenue_by_hotel = cleaned_data.groupby("hotel")["revenue"].sum().sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        plt.bar(revenue_by_hotel.index, revenue_by_hotel.values)
        plt.title("Revenue by Hotel")
        plt.xlabel("Hotel")
        plt.ylabel("Revenue")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(images_folder_path, "revenue_by_hotel.png"))
        plt.show()

        top_hotel = revenue_by_hotel.index[0]
        print("Observation:", top_hotel, "generates the highest revenue.")
    else:
        print("Skipped revenue by hotel chart because hotel or revenue is missing.")

In [ ]:
if booking_data is not None:
    if "average_daily_rate" in cleaned_data.columns:
        plt.figure(figsize=(10, 5))
        plt.hist(cleaned_data["average_daily_rate"], bins=30)
        plt.title("Average Daily Rate Distribution")
        plt.xlabel("Average Daily Rate")
        plt.ylabel("Number of Bookings")
        plt.tight_layout()
        plt.savefig(os.path.join(images_folder_path, "average_daily_rate_distribution.png"))
        plt.show()

        print("Observation: This chart shows whether most bookings are low price, medium price, or high price.")
    else:
        print("Skipped price histogram because average_daily_rate is missing.")

## Step 9: Save Cleaned Data

The cleaned dataset is saved for Power BI. The raw data file is not changed.

In [ ]:
if booking_data is not None:
    if os.path.exists(processed_folder_path) == False:
        os.makedirs(processed_folder_path)

    output_file_path = os.path.join(processed_folder_path, "cleaned_hotel_bookings.csv")
    cleaned_data.to_csv(output_file_path, index=False)

    print("Cleaned data saved here:")
    print(output_file_path)

## Step 10: Power BI Preparation

Use `data/processed/cleaned_hotel_bookings.csv` in Power BI.

Recommended dashboard items:
- KPI cards for total bookings, revenue, average daily rate, cancellation percentage, and occupancy percentage
- Line chart for monthly bookings
- Bar chart for revenue by hotel or city
- Pie chart for room type distribution
- Filters for year, month, hotel, city, room type, and cancellation status